# Extract per-run MILP statistics

Set `csv_path` in the first code cell, then run the notebook. It reads the CSV with pandas, opens each `log_path`, isolates the per-label block matching that row's `adv_label`, and returns an enriched `result_df` without saving a CSV.

In [ ]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path
import math
import re

import pandas as pd

PROJECT_ROOT = Path('/root/Projects/Shuey/Patch-Attack-Verification')

# Set this to the CSV you want to enrich.
csv_path = PROJECT_ROOT / 'csv' / 'recursive_timeout_refinement_20260509_075857.csv'

# Re-running the notebook on an already-enriched CSV is easier if old extracted
# columns are removed before attaching fresh values.
drop_existing_extracted_columns = True

LOG_PATH_COLUMN = 'log_path'
ADV_LABEL_COLUMN = 'adv_label'

COUNTER_PATTERN = re.compile(
    r'Counter is\s+\d+(?:\s+candidate label is\s+(\d+)\s+adv label is\s+(\d+)|\s+label is\s+(\d+))'
)
NUMBER = r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?'
RANGE_RE = re.compile(r'\[\s*([^,\]]+)\s*,\s*([^\]]+)\s*\]')

EXTRACTED_PREFIXES = (
    'extract_',
    'block_',
    'model_stats_',
    'gurobi_',
    'presolve_',
    'root_relaxation_',
    'cutting_planes_',
    'mip_final_',
    'milp_',
)


def clean_key(text: str) -> str:
    key = text.strip().lower()
    key = key.replace('%', 'pct')
    key = re.sub(r'[^a-z0-9]+', '_', key)
    return key.strip('_')


def model_prefix(model_name: str) -> str:
    key = clean_key(model_name)
    if key in {'milp_pre', 'presolved'}:
        return 'model_stats_presolved'
    return f'model_stats_{key}'


def parse_number_token(value: object):
    if value is None:
        return pd.NA
    text = str(value).strip().rstrip(',')
    if text in {'', '-', 'None', 'nan'}:
        return pd.NA
    if text.endswith('%'):
        text = text[:-1]
    try:
        number = float(text)
    except ValueError:
        return text
    if math.isfinite(number) and number.is_integer():
        return int(number)
    return number


def parse_range(text: str):
    match = RANGE_RE.search(text)
    if not match:
        return None
    return parse_number_token(match.group(1)), parse_number_token(match.group(2))


def first_match(pattern: str, text: str, flags: int = 0):
    return re.search(pattern, text, flags)


@lru_cache(maxsize=None)
def read_log_text(log_path: str) -> str:
    return Path(log_path).read_text(errors='replace')


def iter_label_blocks(text: str):
    matches = list(COUNTER_PATTERN.finditer(text))
    if not matches:
        yield {
            'block_index': 0,
            'candidate_label': pd.NA,
            'adv_label': pd.NA,
            'block_text': text,
        }
        return

    for index, match in enumerate(matches):
        candidate_label = int(match.group(1)) if match.group(1) is not None else pd.NA
        adv_label = int(match.group(2) if match.group(2) is not None else match.group(3))
        start = match.start()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        yield {
            'block_index': index,
            'candidate_label': candidate_label,
            'adv_label': adv_label,
            'block_text': text[start:end],
        }


def select_relevant_block(row: pd.Series, text: str) -> tuple[dict, str | None]:
    blocks = list(iter_label_blocks(text))
    if ADV_LABEL_COLUMN not in row or pd.isna(row[ADV_LABEL_COLUMN]):
        if len(blocks) == 1:
            return blocks[0], None
        return {'block_index': pd.NA, 'candidate_label': pd.NA, 'adv_label': pd.NA, 'block_text': ''}, 'missing adv_label; log has multiple label blocks'

    adv_label = int(row[ADV_LABEL_COLUMN])
    matches = [block for block in blocks if block['adv_label'] == adv_label]
    if len(matches) == 1:
        return matches[0], None
    if len(matches) > 1:
        return matches[0], f'multiple blocks matched adv_label={adv_label}; used first match'
    if len(blocks) == 1 and pd.isna(blocks[0]['adv_label']):
        return blocks[0], 'no per-label marker found; used whole log'
    return {'block_index': pd.NA, 'candidate_label': pd.NA, 'adv_label': pd.NA, 'block_text': ''}, f'no block matched adv_label={adv_label}'


def parse_model_print_stats(block_text: str) -> dict[str, object]:
    stats: dict[str, object] = {}
    headers = list(re.finditer(r'Statistics for model\s+([^:\n]+)\s*:', block_text))

    for index, header in enumerate(headers):
        model_name = header.group(1).strip()
        prefix = model_prefix(model_name)
        start = header.end()
        next_header = headers[index + 1].start() if index + 1 < len(headers) else len(block_text)
        none_match = re.search(r'\nNone\b', block_text[start:next_header])
        end = start + none_match.start() if none_match else next_header
        section = block_text[start:end]

        stats[f'{prefix}_model_name'] = model_name
        for raw_line in section.splitlines():
            line = raw_line.strip()
            if not line or ':' not in line:
                continue
            label, value = [part.strip() for part in line.split(':', 1)]
            key = clean_key(label)

            if label == 'Linear constraint matrix':
                match = re.search(rf'({NUMBER})\s+Constrs,\s+({NUMBER})\s+Vars,\s+({NUMBER})\s+NZs', value, re.I)
                if match:
                    stats[f'{prefix}_linear_constrs'] = parse_number_token(match.group(1))
                    stats[f'{prefix}_linear_vars'] = parse_number_token(match.group(2))
                    stats[f'{prefix}_linear_nzs'] = parse_number_token(match.group(3))
                else:
                    stats[f'{prefix}_{key}_raw'] = value
                continue

            if label == 'Variable types':
                match = re.search(rf'({NUMBER})\s+Continuous,\s+({NUMBER})\s+Integer(?:\s+\(({NUMBER})\s+Binary\))?', value, re.I)
                if match:
                    stats[f'{prefix}_continuous_vars'] = parse_number_token(match.group(1))
                    stats[f'{prefix}_integer_vars'] = parse_number_token(match.group(2))
                    stats[f'{prefix}_binary_vars'] = parse_number_token(match.group(3))
                else:
                    stats[f'{prefix}_{key}_raw'] = value
                continue

            if label == 'General constraints':
                match = re.search(rf'({NUMBER})', value)
                stats[f'{prefix}_general_constraints'] = parse_number_token(match.group(1)) if match else value
                continue

            parsed_range = parse_range(value)
            if parsed_range is not None:
                stats[f'{prefix}_{key}_min'] = parsed_range[0]
                stats[f'{prefix}_{key}_max'] = parsed_range[1]
            else:
                stats[f'{prefix}_{key}'] = parse_number_token(value)
    return stats


def parse_gurobi_solve_stats(block_text: str) -> dict[str, object]:
    stats: dict[str, object] = {}

    match = first_match(rf'create_model NumIntVars:\s*({NUMBER})\s+NumBinVars:\s*({NUMBER})\s+use_milp:\s*(\w+)\s+partial_milp:\s*({NUMBER})\s+max_milp_neurons:\s*({NUMBER})', block_text)
    if match:
        stats['milp_create_model_integer_vars'] = parse_number_token(match.group(1))
        stats['milp_create_model_binary_vars'] = parse_number_token(match.group(2))
        stats['milp_create_model_use_milp'] = match.group(3)
        stats['milp_create_model_partial_milp'] = parse_number_token(match.group(4))
        stats['milp_create_model_max_milp_neurons'] = parse_number_token(match.group(5))

    match = first_match(rf'Total constraints \(all types\):\s*({NUMBER})', block_text)
    if match:
        stats['milp_total_constraints_all_types'] = parse_number_token(match.group(1))

    match = first_match(rf'Changed value of parameter Cutoff to\s*({NUMBER})', block_text)
    if match:
        stats['gurobi_cutoff'] = parse_number_token(match.group(1))

    match = first_match(rf'Optimize a model with\s+({NUMBER})\s+rows,\s+({NUMBER})\s+columns and\s+({NUMBER})\s+nonzeros', block_text, re.I)
    if match:
        stats['gurobi_model_rows'] = parse_number_token(match.group(1))
        stats['gurobi_model_columns'] = parse_number_token(match.group(2))
        stats['gurobi_model_nonzeros'] = parse_number_token(match.group(3))

    match = first_match(r'Model fingerprint:\s*(\S+)', block_text)
    if match:
        stats['gurobi_model_fingerprint'] = match.group(1)

    match = first_match(rf'Model has\s+({NUMBER})\s+general constraints', block_text, re.I)
    if match:
        stats['gurobi_model_general_constraints'] = parse_number_token(match.group(1))

    match = first_match(rf'Variable types:\s+({NUMBER})\s+continuous,\s+({NUMBER})\s+integer(?:\s+\(({NUMBER})\s+binary\))?', block_text, re.I)
    if match:
        stats['gurobi_model_continuous_vars'] = parse_number_token(match.group(1))
        stats['gurobi_model_integer_vars'] = parse_number_token(match.group(2))
        stats['gurobi_model_binary_vars'] = parse_number_token(match.group(3))

    coefficient_section = re.search(r'Coefficient statistics:\n(?P<section>.*?)(?:\n\S|\Z)', block_text, re.S)
    if coefficient_section:
        for raw_line in coefficient_section.group('section').splitlines():
            line = raw_line.strip()
            if not line or '[' not in line:
                continue
            label = line.split('[', 1)[0].strip()
            parsed_range = parse_range(line)
            if parsed_range is None:
                continue
            key = clean_key(label)
            stats[f'gurobi_coefficient_{key}_min'] = parsed_range[0]
            stats[f'gurobi_coefficient_{key}_max'] = parsed_range[1]

    presolve_removed = list(re.finditer(rf'Presolve removed\s+({NUMBER})\s+rows and\s+({NUMBER})\s+columns', block_text, re.I))
    if presolve_removed:
        match = presolve_removed[-1]
        stats['presolve_removed_rows'] = parse_number_token(match.group(1))
        stats['presolve_removed_columns'] = parse_number_token(match.group(2))
        stats['presolve_removed_occurrences'] = len(presolve_removed)

    presolve_times = list(re.finditer(rf'Presolve time:\s*({NUMBER})s', block_text, re.I))
    if presolve_times:
        stats['presolve_time_seconds'] = parse_number_token(presolve_times[-1].group(1))
        stats['presolve_time_occurrences'] = len(presolve_times)

    match = first_match(rf'Presolved:\s+({NUMBER})\s+rows,\s+({NUMBER})\s+columns,\s+({NUMBER})\s+nonzeros', block_text, re.I)
    if match:
        stats['presolve_rows'] = parse_number_token(match.group(1))
        stats['presolve_columns'] = parse_number_token(match.group(2))
        stats['presolve_nonzeros'] = parse_number_token(match.group(3))

    presolved_var_matches = list(re.finditer(rf'Variable types:\s+({NUMBER})\s+continuous,\s+({NUMBER})\s+integer(?:\s+\(({NUMBER})\s+binary\))?', block_text, re.I))
    if len(presolved_var_matches) >= 2:
        match = presolved_var_matches[-1]
        stats['presolve_continuous_vars'] = parse_number_token(match.group(1))
        stats['presolve_integer_vars'] = parse_number_token(match.group(2))
        stats['presolve_binary_vars'] = parse_number_token(match.group(3))

    match = first_match(rf'Root relaxation:\s+(objective\s+({NUMBER})|cutoff),\s+({NUMBER})\s+iterations,\s+({NUMBER})\s+seconds', block_text, re.I)
    if match:
        stats['root_relaxation_result'] = 'objective' if match.group(2) is not None else 'cutoff'
        stats['root_relaxation_objective'] = parse_number_token(match.group(2)) if match.group(2) is not None else pd.NA
        stats['root_relaxation_iterations'] = parse_number_token(match.group(3))
        stats['root_relaxation_seconds'] = parse_number_token(match.group(4))

    cutting_section = re.search(r'Cutting planes:\n(?P<section>.*?)(?:\n\s*\n|\Z)', block_text, re.S)
    if cutting_section:
        for raw_line in cutting_section.group('section').splitlines():
            line = raw_line.strip()
            if not line or ':' not in line:
                continue
            label, value = [part.strip() for part in line.split(':', 1)]
            stats[f'cutting_planes_{clean_key(label)}'] = parse_number_token(value)

    match = first_match(rf'Explored\s+({NUMBER})\s+nodes\s+\(({NUMBER})\s+simplex iterations\)\s+in\s+({NUMBER})\s+seconds', block_text, re.I)
    if match:
        stats['gurobi_explored_nodes'] = parse_number_token(match.group(1))
        stats['gurobi_simplex_iterations'] = parse_number_token(match.group(2))

    match = first_match(rf'Solution count\s+({NUMBER})', block_text, re.I)
    if match:
        stats['gurobi_solution_count'] = parse_number_token(match.group(1))

    match = first_match(r'Best objective\s+([^,]+),\s+best bound\s+([^,]+),\s+gap\s+([^\n]+)', block_text, re.I)
    if match:
        stats['gurobi_best_objective'] = parse_number_token(match.group(1))
        stats['gurobi_best_bound'] = parse_number_token(match.group(2))
        stats['gurobi_gap'] = parse_number_token(match.group(3))

    match = first_match(rf'MIP_FINAL\s+status=({NUMBER})\s+runtime=({NUMBER})\s+nodes=({NUMBER})\s+objbound=({NUMBER})\s+solcnt=({NUMBER})', block_text)
    if match:
        stats['mip_final_status'] = parse_number_token(match.group(1))
        stats['mip_final_nodes'] = parse_number_token(match.group(3))
        stats['mip_final_objbound'] = parse_number_token(match.group(4))
        stats['mip_final_solution_count'] = parse_number_token(match.group(5))

    match = first_match(rf'Solcount is:\s*({NUMBER})', block_text)
    if match:
        stats['milp_solcount'] = parse_number_token(match.group(1))

    match = first_match(rf'MILP objbound is\s+({NUMBER})', block_text)
    if match:
        stats['milp_objbound'] = parse_number_token(match.group(1))

    return stats


def extract_statistics_from_block(block_text: str) -> dict[str, object]:
    stats = {}
    stats.update(parse_model_print_stats(block_text))
    stats.update(parse_gurobi_solve_stats(block_text))
    return stats


def extract_statistics_for_row(row: pd.Series) -> dict[str, object]:
    log_path = row.get(LOG_PATH_COLUMN)
    if pd.isna(log_path) or not str(log_path).strip():
        return {}

    log_path = str(log_path).strip()
    if not Path(log_path).exists():
        return {}

    text = read_log_text(log_path)
    block, _warning = select_relevant_block(row, text)
    block_text = block.get('block_text', '')
    if not block_text:
        return {}

    return extract_statistics_from_block(block_text)


def extract_statistics_dataframe(input_csv: Path) -> pd.DataFrame:
    input_csv = Path(input_csv)

    df = pd.read_csv(input_csv)
    if LOG_PATH_COLUMN not in df.columns:
        raise ValueError(f'Input CSV must contain a {LOG_PATH_COLUMN!r} column.')
    if ADV_LABEL_COLUMN not in df.columns:
        raise ValueError(f'Input CSV must contain an {ADV_LABEL_COLUMN!r} column to isolate shared all-label logs.')

    base_df = df.copy()
    if drop_existing_extracted_columns:
        generated_cols = [col for col in base_df.columns if col.startswith(EXTRACTED_PREFIXES)]
        base_df = base_df.drop(columns=generated_cols)

    stats_df = pd.DataFrame([extract_statistics_for_row(row) for _, row in base_df.iterrows()])
    result_df = pd.concat([base_df.reset_index(drop=True), stats_df.reset_index(drop=True)], axis=1)

    extracted_rows = int(stats_df.notna().any(axis=1).sum())
    print(f'Read {len(df)} rows from {input_csv}')
    print(f'Extracted statistics for {extracted_rows} rows')
    return result_df


result_df = extract_statistics_dataframe(Path(csv_path))
result_df.head()
